### Simulando Código Em Produão

In [1]:
# Carregando os pacotes que serão utilizados
import warnings
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier
from category_encoders.one_hot import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, accuracy_score
from config import data

In [2]:
# Carregando o novo arquivo com dados novos
df = pd.read_csv(data/'novos_dados.csv')

In [3]:
# Renomeando as colunas
df.columns = ['ID_Funcionario', 
              'Idade', 
              'Genero', 
              'Anos_Empresa', 
              'Funcao', 
              'Salario_Mensal',
              'Equilibrio_Vida',
              'Satisfacao',
              'Desempenho',
              'Qtd_Promocoes',
              'Hora_Extra',
              'Distancia_Casa',
              'Escolaridade',
              'Estado_Civil',
              'Qtd_Dependentes',
              'Cargo',
              'Tamanho_Empresa',
              'Qtd_Anos_Trabalho',
              'Trabalho_Remoto',
              'Oportunidade_Lideranca',
              'Oportunidade_Inovacao',
              'Reputacao_Empresa',
              'Reconhecimento']              

In [4]:
# Fazendo uma cópia do DataFrame
novos_dados = df.copy()

In [5]:
# Fazendo o tratamento de dados igual fizemos para a criação e treinamento do modelo.
# Importante: Todo tratamento de dado que for realizado nos dados na etapa de construção do modelo devem ser feitos também em dados novos

novos_dados = novos_dados.drop(columns=['ID_Funcionario'])

valores = [-100, 5000, 10000, 20000]
rotulos = ['Até $5.000', 'De $5.001 até $10.000 ', 'Acima de $10.000']
novos_dados['Faixa_Salarial'] = pd.cut(novos_dados['Salario_Mensal'], bins=valores, labels=rotulos)

novos_dados = novos_dados.drop(columns=['Salario_Mensal'])

novos_dados = novos_dados.drop(columns=['Qtd_Anos_Trabalho'])
novos_dados.loc[(novos_dados['Idade'] - novos_dados['Anos_Empresa'] < 18), 'Anos_Empresa'] = 1


variaveis_categoricas_ordinais = ['Faixa_Salarial','Equilibrio_Vida','Satisfacao','Desempenho','Escolaridade','Cargo',
                                 'Tamanho_Empresa','Reputacao_Empresa','Reconhecimento']

encoder = OrdinalEncoder()
for var in variaveis_categoricas_ordinais:
    novos_dados[var] = encoder.fit_transform(novos_dados[var])

variaveis_categoricas_nominais = ['Genero','Funcao','Hora_Extra','Estado_Civil','Trabalho_Remoto','Oportunidade_Lideranca',
                                 'Oportunidade_Inovacao']


encoder = OneHotEncoder(sparse_output=False)
one_hot_encoded = encoder.fit_transform(novos_dados[variaveis_categoricas_nominais])
one_hot_df = pd.DataFrame(one_hot_encoded, columns=encoder.get_feature_names_out(variaveis_categoricas_nominais))
novos_dados = pd.concat([novos_dados, one_hot_df], axis=1)
novos_dados = novos_dados.drop(variaveis_categoricas_nominais, axis=1)

In [6]:
# Visualizando informações do nosso conjunto de dados após as alterações
novos_dados.info()

<class 'pandas.DataFrame'>
RangeIndex: 14900 entries, 0 to 14899
Data columns (total 32 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Idade                       14900 non-null  int64  
 1   Anos_Empresa                14900 non-null  int64  
 2   Equilibrio_Vida             14900 non-null  int64  
 3   Satisfacao                  14900 non-null  int64  
 4   Desempenho                  14900 non-null  int64  
 5   Qtd_Promocoes               14900 non-null  int64  
 6   Distancia_Casa              14900 non-null  int64  
 7   Escolaridade                14900 non-null  int64  
 8   Qtd_Dependentes             14900 non-null  int64  
 9   Cargo                       14900 non-null  int64  
 10  Tamanho_Empresa             14900 non-null  int64  
 11  Reputacao_Empresa           14900 non-null  int64  
 12  Reconhecimento              14900 non-null  int64  
 13  Faixa_Salarial              14900 non-null

In [7]:
# Separando as variaveis preditoras que serão utilizadas no modelo

preditor = novos_dados.copy()

In [8]:
Normalizador = MinMaxScaler()
dados_normalizados = Normalizador.fit_transform(preditor)    

clf = joblib.load(data/'modelo_treinado_rotatividade_funcionarios.pk')


previsoes = clf.predict(dados_normalizados)
probabilidades = clf.predict_proba(dados_normalizados)

df['PREVISOES'] = previsoes
df['PROBABILIDADES'] = probabilidades[:, 1]       

df.to_excel(data/'previsoes_desligamentos.xlsx')
print("Previsoes Geradas com Sucesso!")        

Previsoes Geradas com Sucesso!
